# 🧠 NEUROS Self-Training — 6ced170e-6da
**Base:** unsloth/llama-3-8b-bnb-4bit
**Method:** LoRA (r=16)
**Epochs:** 3
**GPU:** Free T4 (Colab)
**Cost:** $0

In [ ]:
# Install dependencies
!pip install -q unsloth
!pip install -q trl peft datasets transformers accelerate bitsandbytes
from unsloth import FastLanguageModel

In [ ]:
# Load base model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
# Upload and load dataset
from google.colab import files
import json, os

# Option 1: Upload the dataset file
print('Upload evez-alpaca.json:')
uploaded = files.upload()
dataset_name = list(uploaded.keys())[0]

# Format for training
from datasets import load_dataset
dataset = load_dataset('json', data_files=dataset_name, split='train')

alpaca_prompt = """Below is an instruction that describes a task. Write a response.

### Instruction:
{}

### Response:
{}"""

def formatting_func(examples):
    return [alpaca_prompt.format(inst, out) for inst, out in zip(examples['instruction'], examples['output'])]


In [ ]:
# Train!
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    formatting_func = formatting_func,
    max_seq_length = 2048,
    args = TrainingArguments(
        output_dir = "/content/neuros-model",
        num_train_epochs = 3,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        save_steps = 100,
    ),
)
trainer.train()

In [ ]:
# Save model locally
model.save_pretrained('/content/neuros-lora')
tokenizer.save_pretrained('/content/neuros-lora')

# Option A: Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')
model.save_pretrained('/content/drive/MyDrive/neuros-lora')
tokenizer.save_pretrained('/content/drive/MyDrive/neuros-lora')

# Option B: Upload to HuggingFace
# model.push_to_hub_merged("EVEZX/neuros-model", tokenizer)

print('✅ Model trained and saved!')
print('LoRA adapters saved to /content/neuros-lora')